# LeetCode 1586 — Binary Search Tree Iterator II bằng Python

Link đề: https://leetcode.com/problems/binary-search-tree-iterator-ii/ · Độ khó: **Medium** · Chủ đề: Stack, Tree, Design, Binary Search Tree, Binary Tree, Iterator

> **Ghi chú Premium:** đây là bài Premium (🔒). Đề bài và chữ ký dưới đây được viết theo hiểu biết về bài, lấy từ trang lời giải doocs (https://leetcode.doocs.org/en/lc/1586/). Nếu bạn có tài khoản Premium, hãy đối chiếu lại với đề gốc trên LeetCode.

Skeleton LeetCode (bài **thiết kế lớp**):

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class BSTIterator:
    def __init__(self, root: Optional[TreeNode]):
        ...

    def hasNext(self) -> bool:
        ...

    def next(self) -> int:
        ...

    def hasPrev(self) -> bool:
        ...

    def prev(self) -> int:
        ...

# Your BSTIterator object will be instantiated and called as such:
# obj = BSTIterator(root)
# param_1 = obj.hasNext()
# param_2 = obj.next()
# param_3 = obj.hasPrev()
# param_4 = obj.prev()
```

## 1. Mục tiêu học

1. Hiểu **duyệt trung thứ tự (inorder)** của cây nhị phân tìm kiếm (BST) cho dãy giá trị **tăng dần**, và mô hình "con trỏ đứng giữa các phần tử" của iterator hai chiều.
2. Cài đặt baseline: **làm phẳng cây thành mảng** bằng inorder (đệ quy rồi đến **stack tường minh** để tránh tràn đệ quy với cây suy biến `10⁵` nút) + một chỉ số `i`.
3. Trả lời **follow-up** "không tính trước toàn bộ giá trị": iterator **lười** dùng stack sinh phần tử khi cần + mảng lưu các phần tử **đã sinh** để `prev` quay lại.
4. Xử lý chính xác các điều kiện biên của `hasNext` / `hasPrev` khi con trỏ đi tới đi lui.
5. Tự dựng `TreeNode`, hàm dựng cây từ mảng level-order, và **bộ chạy chuỗi lệnh giả lập** kiểu LeetCode để kiểm thử ngẫu nhiên với seed cố định.

## 2. Đề bài & ví dụ

Cài đặt lớp `BSTIterator` biểu diễn một iterator trên **dãy duyệt trung thứ tự** của một BST:

- `BSTIterator(TreeNode root)`: khởi tạo. Con trỏ được đặt ở một **số không tồn tại nhỏ hơn mọi phần tử** của BST.
- `boolean hasNext()`: trả `true` nếu **bên phải** con trỏ còn phần tử.
- `int next()`: dịch con trỏ sang phải **một bước** rồi trả về số tại con trỏ.
- `boolean hasPrev()`: trả `true` nếu **bên trái** con trỏ còn phần tử.
- `int prev()`: dịch con trỏ sang trái một bước rồi trả về số tại con trỏ.

Vì con trỏ khởi đầu ở "số nhỏ hơn mọi phần tử", lần gọi `next()` đầu tiên trả về phần tử **nhỏ nhất**. Các lời gọi `next()`/`prev()` luôn **hợp lệ** (có phần tử để đi tới).

**Ví dụ:**
```
Input
["BSTIterator", "next", "next", "prev", "next", "hasNext", "next", "next", "next", "hasNext", "hasPrev", "prev", "prev"]
[[[7, 3, 15, null, null, 9, 20]], [], [], [], [], [], [], [], [], [], [], [], []]
Output
[null, 3, 7, 3, 7, true, 9, 15, 20, false, true, 15, 9]
```
Dãy inorder là `[3, 7, 9, 15, 20]`.

**Ràng buộc:**
- Số nút trong `[1, 10⁵]`.
- `0 ≤ Node.val ≤ 10⁶`.
- Tối đa `10⁵` lời gọi `hasNext`, `next`, `hasPrev`, `prev`.

**Follow-up:** Có thể giải mà **không tính trước** toàn bộ giá trị của cây không?

In [1]:
import random
import time
from collections import deque
from typing import List, Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values: List[Optional[int]]) -> Optional[TreeNode]:
    """Dựng cây từ mảng level-order kiểu LeetCode (None = không có nút)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            q.append(node.right)
        i += 1
    return root

EX_CMDS = ["BSTIterator", "next", "next", "prev", "next", "hasNext", "next", "next", "next", "hasNext", "hasPrev", "prev", "prev"]
EX_ARGS = [[[7, 3, 15, None, None, 9, 20]], [], [], [], [], [], [], [], [], [], [], [], []]
EX_OUT = [None, 3, 7, 3, 7, True, 9, 15, 20, False, True, 15, 9]
root = build_tree([7, 3, 15, None, None, 9, 20])
print("gốc:", root.val, "| con trái:", root.left.val, "| con phải:", root.right.val, root.right.left.val, root.right.right.val)

gốc: 7 | con trái: 3 | con phải: 15 9 20


## 3. Kiến thức nền

### 3.1 Cây nhị phân tìm kiếm (BST) và duyệt inorder

BST: mọi nút trong cây con trái **nhỏ hơn** nút, mọi nút trong cây con phải **lớn hơn**. Duyệt **trái → nút → phải** (inorder) cho dãy **tăng dần**.

In [2]:
def inorder_recursive(node, out):
    if node is None:
        return
    inorder_recursive(node.left, out)
    out.append(node.val)
    inorder_recursive(node.right, out)

vals = []
inorder_recursive(root, vals)
print("inorder:", vals, "| tăng dần?", vals == sorted(vals))

inorder: [3, 7, 9, 15, 20] | tăng dần? True


### 3.2 Giới hạn đệ quy của Python

Python mặc định chỉ cho đệ quy sâu ~1000 tầng. Cây suy biến (mỗi nút chỉ có con trái) với `10⁵` nút làm hàm đệ quy ném `RecursionError`. Ta **không** tăng giới hạn (có thể làm sập tiến trình); thay vào đó dùng **stack tường minh**.

In [3]:
import sys
print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

def left_chain(n):
    """Cây suy biến: n, n-1, ..., 1 nối qua con trái (inorder = 1..n)."""
    root = None
    for v in range(1, n + 1):
        root = TreeNode(v, left=root)
    return root

deep = left_chain(5000)
try:
    inorder_recursive(deep, [])
    print("đệ quy chạy được")
except RecursionError as e:
    print("RecursionError:", str(e)[:60])

giới hạn đệ quy hiện tại: 1000
RecursionError: maximum recursion depth exceeded


### 3.3 Inorder bằng stack: "đẩy hết nhánh trái"

Mẫu kinh điển: đẩy nút hiện tại và **toàn bộ chuỗi con trái** vào stack; pop ra nút trên cùng (nhỏ nhất chưa thăm), ghi nhận, rồi chuyển sang cây con phải và lại đẩy hết nhánh trái của nó.

In [4]:
def inorder_iterative(root):
    out, stack, node = [], [], root
    while stack or node:
        while node:                 # đẩy hết nhánh trái
            stack.append(node)
            node = node.left
        node = stack.pop()          # nút nhỏ nhất chưa thăm
        out.append(node.val)
        node = node.right           # chuyển sang cây con phải
    return out

print(inorder_iterative(root))
print("cây suy biến 5000 nút, 5 giá trị đầu:", inorder_iterative(deep)[:5])

[3, 7, 9, 15, 20]
cây suy biến 5000 nút, 5 giá trị đầu: [1, 2, 3, 4, 5]


### 3.4 Mô hình con trỏ của iterator hai chiều

Đánh số các phần tử inorder `0..n-1`. Con trỏ là một chỉ số `i`, ban đầu `i = -1` ("trước phần tử đầu").
- `next()`: `i += 1`, trả `a[i]`. `hasNext()` ⇔ `i < n - 1`.
- `prev()`: `i -= 1`, trả `a[i]`. `hasPrev()` ⇔ `i > 0` (phải có phần tử **bên trái** phần tử hiện tại).

In [5]:
a = [3, 7, 9, 15, 20]
i = -1
log = []
for op in ["next", "next", "prev", "next", "next"]:
    if op == "next":
        i += 1
    else:
        i -= 1
    log.append((op, a[i], i))
print(log)
print("hasNext:", i < len(a) - 1, "| hasPrev:", i > 0)

[('next', 3, 0), ('next', 7, 1), ('prev', 3, 0), ('next', 7, 1), ('next', 9, 2)]
hasNext: True | hasPrev: True


### 3.5 Lớp và phương thức — bộ chạy chuỗi lệnh

LeetCode mô tả test bằng hai mảng: tên lệnh và đối số. Ta viết hàm `run_commands(cls, cmds, args)` gọi `getattr(obj, tên)(*đối số)` và gom kết quả — **đây là bản giả lập** để test cục bộ.

In [6]:
def run_commands(cls, cmds, args):
    out, obj = [], None
    for name, a in zip(cmds, args):
        if name == "BSTIterator":
            obj = cls(build_tree(a[0]))
            out.append(None)
        else:
            out.append(getattr(obj, name)(*a))
    return out

class Dummy:
    def __init__(self, root):
        self.v = root.val
    def hasNext(self):
        return False
print(run_commands(Dummy, ["BSTIterator", "hasNext"], [[[5]], []]))

[None, False]


## 4. Ý tưởng nền tảng / trực giác

Cây `[7, 3, 15, null, null, 9, 20]`:

```
        7
       / \
      3   15
         /  \
        9    20
```

Dãy inorder: `[3, 7, 9, 15, 20]`. Chạy tay chuỗi lệnh của ví dụ, con trỏ `i` (gạch chân):

| lệnh | i sau lệnh | trạng thái | trả về |
|---|---|---|---|
| khởi tạo | -1 | `^ 3 7 9 15 20` | null |
| next | 0 | `[3] 7 9 15 20` | 3 |
| next | 1 | `3 [7] 9 15 20` | 7 |
| prev | 0 | `[3] 7 9 15 20` | 3 |
| next | 1 | `3 [7] 9 15 20` | 7 |
| hasNext | 1 | còn 9, 15, 20 bên phải | true |
| next ×3 | 4 | `3 7 9 15 [20]` | 9, 15, 20 |
| hasNext | 4 | không còn bên phải | false |
| hasPrev | 4 | còn bên trái | true |
| prev ×2 | 2 | `3 7 [9] 15 20` | 15, 9 |

Nhận xét: nếu có sẵn mảng inorder, mọi thao tác chỉ là `i ± 1` và so sánh biên. Câu hỏi follow-up là làm sao **không** tạo cả mảng ngay từ đầu.

## 5. Cách 1 — Brute Force / baseline

**Ý tưởng ngây thơ nhất:** chỉ lưu `root` và chỉ số `i`. Mỗi lần gọi `next`/`prev`/`hasNext`/`hasPrev` lại **duyệt inorder toàn bộ cây** để lấy phần tử thứ `i`.

Đúng nhưng mỗi thao tác tốn `O(n)` ⇒ `10⁵` lời gọi × `10⁵` nút = `10¹⁰`. Ta dùng nó làm **oracle** cho bộ test (với cây nhỏ).

In [7]:
class BSTIteratorBrute:
    """Baseline: duyệt lại cả cây ở mỗi lời gọi, O(n) mỗi thao tác."""

    def __init__(self, root: Optional[TreeNode]):
        self.root = root
        self.i = -1

    def _all(self):
        return inorder_iterative(self.root)

    def hasNext(self) -> bool:
        return self.i < len(self._all()) - 1

    def next(self) -> int:
        self.i += 1
        return self._all()[self.i]

    def hasPrev(self) -> bool:
        return self.i > 0

    def prev(self) -> int:
        self.i -= 1
        return self._all()[self.i]

got = run_commands(BSTIteratorBrute, EX_CMDS, EX_ARGS)
print(got)
print("khớp ví dụ:", got == EX_OUT)

[None, 3, 7, 3, 7, True, 9, 15, 20, False, True, 15, 9]
khớp ví dụ: True


**Độ phức tạp:** khởi tạo `O(1)`; mỗi thao tác `O(n)` thời gian và `O(n)` bộ nhớ tạm.

**Nút thắt:** duyệt lại cả cây ở mỗi lời gọi. Cải tiến hiển nhiên: duyệt **một lần** lúc khởi tạo và lưu lại (cách 2), hoặc chỉ sinh **dần** những phần tử cần dùng (cách 4).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — tách "sinh phần tử" khỏi "di chuyển con trỏ":** iterator hai chiều = (một nguồn sinh phần tử inorder **theo thứ tự**) + (một **bộ đệm** các phần tử đã sinh) + (chỉ số `i` trong bộ đệm).

**Insight 2 — `prev` không bao giờ cần phần tử mới:** `prev` chỉ đi lùi về những phần tử **đã từng** được `next` trả về ⇒ luôn nằm trong bộ đệm.

**Insight 3 — `next` có hai trường hợp:**
- *nếu* `i < len(buf) - 1` (con trỏ đã lùi trước đó) *thì* chỉ cần `i += 1` và trả `buf[i]`;
- *ngược lại* phải **sinh phần tử inorder kế tiếp** (stack "đẩy hết nhánh trái"), thêm vào `buf`, `i += 1`.

**Insight 4 — `hasNext` cũng có hai nguồn:** còn phần tử trong bộ đệm phía sau con trỏ **hoặc** stack còn nút chưa sinh.

Bộ đệm có thể chứa tới `n` phần tử nếu người dùng duyệt hết cây, nhưng ta chỉ trả chi phí cho những gì thực sự dùng. Stack chỉ giữ `O(h)` nút (`h` = chiều cao cây).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Inorder đệ quy vào mảng + con trỏ (doocs, lời giải 1)

**Khởi tạo:** `dfs` đệ quy đẩy giá trị vào `self.nums`; `self.i = -1`.

**Thao tác:** đúng như mô hình con trỏ ở phần 3.4.

In [8]:
class BSTIteratorArrayRec:
    def __init__(self, root: Optional[TreeNode]):
        self.nums = []

        def dfs(node):
            if node is None:
                return
            dfs(node.left)
            self.nums.append(node.val)
            dfs(node.right)

        dfs(root)
        self.i = -1

    def hasNext(self) -> bool:
        return self.i < len(self.nums) - 1

    def next(self) -> int:
        self.i += 1
        return self.nums[self.i]

    def hasPrev(self) -> bool:
        return self.i > 0

    def prev(self) -> int:
        self.i -= 1
        return self.nums[self.i]

print(run_commands(BSTIteratorArrayRec, EX_CMDS, EX_ARGS) == EX_OUT)

True


Code nộp LeetCode cho cách 2 (đúng như doocs lời giải 1):

```python
class BSTIterator:
    def __init__(self, root: Optional[TreeNode]):
        self.nums = []

        def dfs(root):
            if root is None:
                return
            dfs(root.left)
            self.nums.append(root.val)
            dfs(root.right)

        dfs(root)
        self.i = -1

    def hasNext(self) -> bool:
        return self.i < len(self.nums) - 1

    def next(self) -> int:
        self.i += 1
        return self.nums[self.i]

    def hasPrev(self) -> bool:
        return self.i > 0

    def prev(self) -> int:
        self.i -= 1
        return self.nums[self.i]
```

**Độ phức tạp:** khởi tạo `O(n)`, mỗi thao tác `O(1)`, bộ nhớ `O(n)` + `O(h)` stack đệ quy.

**Rủi ro:** cây suy biến sâu `10⁵` tầng ⇒ vượt giới hạn đệ quy mặc định của Python (doocs cũng ghi chú điều này ở lời giải 2).

### 7.2 Cách 3 — Inorder bằng stack tường minh vào mảng (doocs, lời giải 2)

Doocs dùng stack chứa cặp `(nút, trạng thái)`: trạng thái `0` = "vừa vào nút" (đẩy lại nút với trạng thái `1`, rồi đẩy con trái); trạng thái `1` = "đã xong cây con trái" (ghi giá trị, đẩy con phải). Cách này mô phỏng đúng khung đệ quy, không bị giới hạn độ sâu.

In [9]:
class BSTIteratorArrayStack:
    def __init__(self, root: Optional[TreeNode]):
        self.nums = []
        stk = [(root, 0)]
        while stk:
            node, state = stk.pop()
            if node is None:
                continue
            if state == 0:
                stk.append((node, 1))
                stk.append((node.left, 0))
                continue
            self.nums.append(node.val)
            stk.append((node.right, 0))
        self.i = -1

    def hasNext(self) -> bool:
        return self.i < len(self.nums) - 1

    def next(self) -> int:
        self.i += 1
        return self.nums[self.i]

    def hasPrev(self) -> bool:
        return self.i > 0

    def prev(self) -> int:
        self.i -= 1
        return self.nums[self.i]

print(run_commands(BSTIteratorArrayStack, EX_CMDS, EX_ARGS) == EX_OUT)
it = BSTIteratorArrayStack(left_chain(10**5))
print("cây suy biến 10^5 nút:", it.next(), it.next(), it.prev(), "| hasPrev:", it.hasPrev())

True
cây suy biến 10^5 nút: 1 2 1 | hasPrev: False


Code nộp LeetCode cho cách 3 (đúng như doocs lời giải 2):

```python
class BSTIterator:
    def __init__(self, root: Optional[TreeNode]):
        self.nums = []
        stk = [(root, 0)]
        while stk:
            node, state = stk.pop()
            if node is None:
                continue
            if state == 0:
                stk.append((node, 1))
                stk.append((node.left, 0))
                continue
            self.nums.append(node.val)
            stk.append((node.right, 0))
        self.i = -1

    def hasNext(self) -> bool:
        return self.i < len(self.nums) - 1

    def next(self) -> int:
        self.i += 1
        return self.nums[self.i]

    def hasPrev(self) -> bool:
        return self.i > 0

    def prev(self) -> int:
        self.i -= 1
        return self.nums[self.i]
```

**Độ phức tạp:** khởi tạo `O(n)`, mỗi thao tác `O(1)`, bộ nhớ `O(n)` (mảng) + `O(h)` (stack lúc khởi tạo).

### 7.3 Cách 4 — Follow-up: iterator lười (stack + mảng đã duyệt)

**Trạng thái:**
- `self.stack`: stack inorder "đẩy hết nhánh trái" (giống LC 173) — nguồn sinh phần tử mới.
- `self.arr`: các giá trị **đã sinh** theo thứ tự.
- `self.i`: con trỏ trong `arr`, ban đầu `-1`.

**`next()`:**
- nếu `i < len(arr) - 1` (đang ở "vùng đã duyệt" do từng `prev`): `i += 1`, trả `arr[i]`;
- ngược lại: pop nút trên cùng của stack, đẩy nhánh trái của cây con phải, thêm giá trị vào `arr`, `i += 1`.

**`hasNext()`:** `i < len(arr) - 1` **hoặc** stack khác rỗng.

**`prev()` / `hasPrev()`:** như mảng (`i -= 1`; `i > 0`).

Không cần tính trước toàn bộ cây; chỉ sinh khi thực sự đi tới phần tử mới.

In [10]:
class BSTIteratorLazy:
    def __init__(self, root: Optional[TreeNode]):
        self.stack = []
        self._push_left(root)
        self.arr = []
        self.i = -1

    def _push_left(self, node):
        while node:
            self.stack.append(node)
            node = node.left

    def hasNext(self) -> bool:
        return self.i < len(self.arr) - 1 or bool(self.stack)

    def next(self) -> int:
        self.i += 1
        if self.i == len(self.arr):          # cần sinh phần tử mới
            node = self.stack.pop()
            self._push_left(node.right)
            self.arr.append(node.val)
        return self.arr[self.i]

    def hasPrev(self) -> bool:
        return self.i > 0

    def prev(self) -> int:
        self.i -= 1
        return self.arr[self.i]

print(run_commands(BSTIteratorLazy, EX_CMDS, EX_ARGS) == EX_OUT)
it = BSTIteratorLazy(left_chain(10**5))
print("lười trên cây suy biến:", it.next(), it.next(), it.prev(), "| đã sinh", len(it.arr), "phần tử")

True
lười trên cây suy biến: 1 2 1 | đã sinh 2 phần tử


Code nộp LeetCode cho cách 4 (follow-up, giống phần 9):

```python
class BSTIterator:
    def __init__(self, root: Optional[TreeNode]):
        self.stack = []
        self._push_left(root)
        self.arr = []
        self.i = -1

    def _push_left(self, node):
        while node:
            self.stack.append(node)
            node = node.left

    def hasNext(self) -> bool:
        return self.i < len(self.arr) - 1 or bool(self.stack)

    def next(self) -> int:
        self.i += 1
        if self.i == len(self.arr):
            node = self.stack.pop()
            self._push_left(node.right)
            self.arr.append(node.val)
        return self.arr[self.i]

    def hasPrev(self) -> bool:
        return self.i > 0

    def prev(self) -> int:
        self.i -= 1
        return self.arr[self.i]
```

**Độ phức tạp:** khởi tạo `O(h)`; `next` **khấu hao** `O(1)` (mỗi nút được đẩy và pop đúng một lần trong suốt quá trình), xấu nhất một lần `O(h)`; `prev`, `hasNext`, `hasPrev` `O(1)`. Bộ nhớ `O(h + k)` với `k` = số phần tử đã sinh (`≤ n`).

In [11]:
# So sánh thời gian khởi tạo trên cây lớn (cân bằng ngẫu nhiên 10^5 nút), rồi chỉ gọi vài lệnh
def random_bst(values):
    """Chèn lần lượt (không đệ quy) để tạo BST."""
    root = None
    for v in values:
        node = TreeNode(v)
        if root is None:
            root = node
            continue
        cur = root
        while True:
            if v < cur.val:
                if cur.left is None:
                    cur.left = node
                    break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = node
                    break
                cur = cur.right
    return root

rng = random.Random(1586)
big_vals = rng.sample(range(10**6 + 1), 10**5)
big_root = random_bst(big_vals)
for cls in [BSTIteratorArrayStack, BSTIteratorLazy]:
    t0 = time.perf_counter()
    it = cls(big_root)
    first = [it.next() for _ in range(3)]
    print(f"{cls.__name__:<22} khởi tạo + 3 lần next: {time.perf_counter() - t0:.4f}s -> {first}")
assert first == sorted(big_vals)[:3]

BSTIteratorArrayStack  khởi tạo + 3 lần next: 0.0423s -> [8, 9, 20]
BSTIteratorLazy        khởi tạo + 3 lần next: 0.0009s -> [8, 9, 20]


## 8. TRACE từng bước

Cách 4 (lười) trên cây ví dụ. Ký hiệu stack từ đáy → đỉnh.

| lệnh | xử lý | stack sau | arr sau | i | trả |
|---|---|---|---|---|---|
| khởi tạo | đẩy nhánh trái 7, 3 | `[7, 3]` | `[]` | -1 | null |
| next | i=0 = len(arr) → pop 3, (3 không có con phải) | `[7]` | `[3]` | 0 | 3 |
| next | pop 7, đẩy nhánh trái của 15: 15, 9 | `[15, 9]` | `[3,7]` | 1 | 7 |
| prev | i=0 | `[15, 9]` | `[3,7]` | 0 | 3 |
| next | i=1 < len(arr) → dùng lại | `[15, 9]` | `[3,7]` | 1 | 7 |
| hasNext | stack khác rỗng | | | 1 | true |
| next | pop 9 | `[15]` | `[3,7,9]` | 2 | 9 |
| next | pop 15, đẩy 20 | `[20]` | `[…,15]` | 3 | 15 |
| next | pop 20 | `[]` | `[…,20]` | 4 | 20 |
| hasNext | i = len-1 và stack rỗng | | | 4 | false |
| hasPrev | i > 0 | | | 4 | true |
| prev, prev | | | | 2 | 15, 9 |

In [12]:
class BSTIteratorLazyTrace(BSTIteratorLazy):
    def _state(self):
        return f"stack={[n.val for n in self.stack]} arr={self.arr} i={self.i}"

def trace_lazy(cmds, args):
    obj = None
    for name, a in zip(cmds, args):
        if name == "BSTIterator":
            obj = BSTIteratorLazyTrace(build_tree(a[0]))
            print(f"{'khởi tạo':<10} -> {'null':<6} | {obj._state()}")
            continue
        reuse = name == "next" and obj.i + 1 < len(obj.arr)
        res = getattr(obj, name)()
        note = " (dùng lại arr)" if reuse else ""
        print(f"{name:<10} -> {str(res):<6} | {obj._state()}{note}")

trace_lazy(EX_CMDS, EX_ARGS)

khởi tạo   -> null   | stack=[7, 3] arr=[] i=-1
next       -> 3      | stack=[7] arr=[3] i=0
next       -> 7      | stack=[15, 9] arr=[3, 7] i=1
prev       -> 3      | stack=[15, 9] arr=[3, 7] i=0
next       -> 7      | stack=[15, 9] arr=[3, 7] i=1 (dùng lại arr)
hasNext    -> True   | stack=[15, 9] arr=[3, 7] i=1
next       -> 9      | stack=[15] arr=[3, 7, 9] i=2
next       -> 15     | stack=[20] arr=[3, 7, 9, 15] i=3
next       -> 20     | stack=[] arr=[3, 7, 9, 15, 20] i=4
hasNext    -> False  | stack=[] arr=[3, 7, 9, 15, 20] i=4
hasPrev    -> True   | stack=[] arr=[3, 7, 9, 15, 20] i=4
prev       -> 15     | stack=[] arr=[3, 7, 9, 15, 20] i=3
prev       -> 9      | stack=[] arr=[3, 7, 9, 15, 20] i=2


## 9. Code nộp LeetCode

Đúng tên lớp/phương thức theo doocs: `BSTIterator(root)`, `hasNext`, `next`, `hasPrev`, `prev`. Bản dưới là cách 4 (follow-up, không tính trước); cách 2/3 cũng được chấp nhận.

In [13]:
class BSTIterator:
    def __init__(self, root: Optional[TreeNode]):
        self.stack = []
        self._push_left(root)
        self.arr = []
        self.i = -1

    def _push_left(self, node):
        while node:
            self.stack.append(node)
            node = node.left

    def hasNext(self) -> bool:
        return self.i < len(self.arr) - 1 or bool(self.stack)

    def next(self) -> int:
        self.i += 1
        if self.i == len(self.arr):
            node = self.stack.pop()
            self._push_left(node.right)
            self.arr.append(node.val)
        return self.arr[self.i]

    def hasPrev(self) -> bool:
        return self.i > 0

    def prev(self) -> int:
        self.i -= 1
        return self.arr[self.i]

out = run_commands(BSTIterator, EX_CMDS, EX_ARGS)
print(out)
print("khớp ví dụ:", out == EX_OUT)

[None, 3, 7, 3, 7, True, 9, 15, 20, False, True, 15, 9]
khớp ví dụ: True


## 10. Vì sao thuật toán đúng

**Cách 2/3.** Inorder của BST là dãy tăng dần — đúng thứ tự iterator cần. Bất biến: `0 ≤ i+1 ≤ n` và con trỏ đứng ở `nums[i]` (hoặc "trước phần tử đầu" khi `i = -1`). `next` hợp lệ nên `i + 1 ≤ n - 1`; `prev` hợp lệ nên `i - 1 ≥ 0`. `hasNext` ⇔ tồn tại chỉ số `> i` ⇔ `i < n - 1`; `hasPrev` ⇔ tồn tại chỉ số `< i` **là phần tử thật** ⇔ `i - 1 ≥ 0` ⇔ `i > 0`.

**Cách 4.** Bất biến:
1. `arr` là **tiền tố** của dãy inorder (độ dài `k`).
2. Stack đúng là trạng thái của thuật toán inorder lặp sau khi đã sinh `k` phần tử: đỉnh stack (nếu có) là phần tử inorder thứ `k`; stack rỗng ⇔ `k = n`.
3. `-1 ≤ i ≤ k - 1`.

`next`: nếu `i + 1 < k` thì phần tử kế là `arr[i+1]` (đã có). Nếu `i + 1 = k` thì pop đỉnh stack cho phần tử inorder thứ `k` (theo bất biến 2), đẩy nhánh trái của cây con phải giữ bất biến 2 cho `k + 1`. `hasNext` ⇔ còn phần tử sau `i` ⇔ (`i < k - 1`) hoặc (`k < n` ⇔ stack khác rỗng). `prev`/`hasPrev` chỉ dùng `arr` vì mọi phần tử bên trái con trỏ đều đã sinh.

**Khấu hao.** Mỗi nút được `push` đúng một lần và `pop` đúng một lần trong toàn bộ vòng đời ⇒ tổng chi phí các `next` sinh phần tử mới là `O(n)`.

## 11. Phân tích Time / Space

Gọi `n` = số nút, `h` = chiều cao cây (`log n ≤ h ≤ n`), `q` = số lời gọi, `k` = số phần tử đã sinh.

| Cách | Khởi tạo | next / prev | hasNext / hasPrev | Bộ nhớ |
|---|---|---|---|---|
| 1. Brute force | `O(1)` | `O(n)` | `O(n)` / `O(1)` | `O(n)` tạm mỗi lần |
| 2. Mảng (đệ quy) | `O(n)` | `O(1)` | `O(1)` | `O(n)` + đệ quy `O(h)` (rủi ro tràn) |
| 3. Mảng (stack) | `O(n)` | `O(1)` | `O(1)` | `O(n)` |
| 4. Lười | `O(h)` | khấu hao `O(1)`, xấu nhất `O(h)` / `O(1)` | `O(1)` | `O(h + k)` |

Tổng cho `q` lời gọi: cách 2/3 là `O(n + q)`; cách 4 là `O(h + q + k) ≤ O(n + q)` và **nhỏ hơn nhiều** khi chỉ duyệt một phần cây.

## 12. Bộ test

### 12.1 Ví dụ đề cho mọi cài đặt (bộ chạy chuỗi lệnh giả lập)

In [14]:
ALL = [BSTIteratorBrute, BSTIteratorArrayRec, BSTIteratorArrayStack, BSTIteratorLazy, BSTIterator]
for cls in ALL:
    assert run_commands(cls, EX_CMDS, EX_ARGS) == EX_OUT, cls.__name__
print("Ví dụ đề: OK cho", len(ALL), "cài đặt")

# Edge case: cây 1 nút
cmds = ["BSTIterator", "hasPrev", "hasNext", "next", "hasNext", "hasPrev"]
args = [[[5]], [], [], [], [], []]
for cls in ALL:
    assert run_commands(cls, cmds, args) == [None, False, True, 5, False, False], cls.__name__
print("Cây 1 nút: OK")

Ví dụ đề: OK cho 5 cài đặt
Cây 1 nút: OK


In [15]:
# 12.2 Chuỗi lệnh ngẫu nhiên hợp lệ, seed cố định, đối chiếu với bản brute force
def level_order(root):
    """Chuyển cây về mảng level-order kiểu LeetCode (để truyền qua bộ chạy lệnh)."""
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        q.append(node.left)
        q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def random_commands(n_nodes, n_ops, rng):
    vals = rng.sample(range(0, 50), n_nodes)
    data = level_order(random_bst(vals))
    cmds, args = ["BSTIterator"], [[data]]
    i = -1
    for _ in range(n_ops):
        choices = ["hasNext", "hasPrev"]
        if i < n_nodes - 1:
            choices += ["next"] * 3
        if i > 0:
            choices += ["prev"] * 2
        op = rng.choice(choices)
        if op == "next":
            i += 1
        elif op == "prev":
            i -= 1
        cmds.append(op)
        args.append([])
    return cmds, args

rng = random.Random(2024)
for _ in range(600):
    cmds, args = random_commands(rng.randint(1, 15), rng.randint(1, 40), rng)
    want = run_commands(BSTIteratorBrute, cmds, args)
    for cls in ALL[1:]:
        assert run_commands(cls, cmds, args) == want, (cls.__name__, cmds, args)
print("600 chuỗi lệnh ngẫu nhiên: mọi cài đặt khớp brute force")

600 chuỗi lệnh ngẫu nhiên: mọi cài đặt khớp brute force


In [16]:
# 12.3 Kiểm tra trên cây lớn và cây suy biến (đúng ràng buộc: 10^5 nút, 10^5 lời gọi)
def stress(cls, root, sorted_vals, q, rng):
    it = cls(root)
    i = -1
    for _ in range(q):
        r = rng.random()
        if r < 0.55 and i < len(sorted_vals) - 1:
            i += 1
            assert it.next() == sorted_vals[i]
        elif r < 0.8 and i > 0:
            i -= 1
            assert it.prev() == sorted_vals[i]
        else:
            assert it.hasNext() == (i < len(sorted_vals) - 1)
            assert it.hasPrev() == (i > 0)

sv = sorted(big_vals)
for cls in [BSTIteratorArrayStack, BSTIteratorLazy, BSTIterator]:
    t0 = time.perf_counter()
    stress(cls, big_root, sv, 10**5, random.Random(1))
    print(f"{cls.__name__:<22} cây ngẫu nhiên 10^5 nút, 10^5 lệnh: {time.perf_counter() - t0:.3f}s")

chain = left_chain(10**5)
for cls in [BSTIteratorArrayStack, BSTIteratorLazy]:
    stress(cls, chain, list(range(1, 10**5 + 1)), 10**5, random.Random(2))
print("Cây suy biến 10^5 nút: OK (không đệ quy)")

BSTIteratorArrayStack  cây ngẫu nhiên 10^5 nút, 10^5 lệnh: 0.060s
BSTIteratorLazy        cây ngẫu nhiên 10^5 nút, 10^5 lệnh: 0.030s


BSTIterator            cây ngẫu nhiên 10^5 nút, 10^5 lệnh: 0.031s


Cây suy biến 10^5 nút: OK (không đệ quy)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — `hasPrev` trả `i >= 0`

Khi con trỏ đang ở phần tử **đầu tiên** (`i = 0`), bên trái không còn phần tử thật nào (vị trí `-1` chỉ là "số không tồn tại").

In [17]:
class WrongHasPrev(BSTIteratorArrayStack):
    def hasPrev(self) -> bool:
        return self.i >= 0                     # SAI: phải là i > 0

cmds = ["BSTIterator", "next", "hasPrev"]
args = [[[7, 3, 15, None, None, 9, 20]], [], []]
print("sai:", run_commands(WrongHasPrev, cmds, args), "| đúng:", run_commands(BSTIterator, cmds, args))

sai: [None, 3, True] | đúng: [None, 3, False]


### Lỗi 2 — Bản lười: `hasNext` chỉ nhìn stack

Sau vài lần `prev`, các phần tử phía trước con trỏ nằm trong `arr` chứ không trong stack. Khi stack đã rỗng (đã sinh hết), `hasNext` trả sai `False`.

In [18]:
class WrongLazyHasNext(BSTIteratorLazy):
    def hasNext(self) -> bool:
        return bool(self.stack)                # SAI: quên phần tử đã sinh phía sau con trỏ

cmds = ["BSTIterator", "next", "next", "next", "prev", "hasNext"]
args = [[[2, 1, 3]], [], [], [], [], []]
print("sai:", run_commands(WrongLazyHasNext, cmds, args), "| đúng:", run_commands(BSTIterator, cmds, args))

sai: [None, 1, 2, 3, 2, False] | đúng: [None, 1, 2, 3, 2, True]


### Lỗi 3 — Bản lười: `next` luôn pop stack, không dùng lại `arr` sau `prev`

Sau `prev`, `next` phải trả lại phần tử đã sinh, không được sinh phần tử mới (sẽ "nhảy cóc"). Ở ví dụ dưới, sau `prev` về 3, bản sai pop luôn phần tử 9 từ stack nên trả `9` thay vì `7`.

In [19]:
class WrongLazyNext(BSTIteratorLazy):
    def next(self) -> int:
        node = self.stack.pop()                # SAI: không kiểm tra i < len(arr) - 1
        self._push_left(node.right)
        self.arr.append(node.val)
        self.i = len(self.arr) - 1
        return node.val

cmds = ["BSTIterator", "next", "next", "prev", "next"]
args = [[[7, 3, 15, None, None, 9, 20]], [], [], [], []]
print("sai:", run_commands(WrongLazyNext, cmds, args), "| đúng:", run_commands(BSTIterator, cmds, args))

sai: [None, 3, 7, 3, 9] | đúng: [None, 3, 7, 3, 7]


### Lỗi 4 — Khởi tạo con trỏ `i = 0`

Đề nói con trỏ bắt đầu ở "số nhỏ hơn mọi phần tử", nên lần `next()` đầu phải trả phần tử nhỏ nhất. Với `i = 0`, `next()` đầu trả phần tử thứ hai.

In [20]:
class WrongStart(BSTIteratorArrayStack):
    def __init__(self, root):
        super().__init__(root)
        self.i = 0                             # SAI: phải là -1

cmds = ["BSTIterator", "next", "next"]
args = [[[7, 3, 15, None, None, 9, 20]], [], []]
print("sai:", run_commands(WrongStart, cmds, args), "| đúng:", run_commands(BSTIterator, cmds, args))

sai: [None, 7, 9] | đúng: [None, 3, 7]


### Lỗi 5 — Dùng DFS đệ quy trên cây suy biến sâu

Cách 2 đúng về logic, nhưng với cây suy biến `10⁵` tầng, Python ném `RecursionError` (ta bắt lại để in).

In [21]:
try:
    BSTIteratorArrayRec(left_chain(10**5))
    print("chạy được (không mong đợi)")
except RecursionError as e:
    print("Cách 2 (đệ quy) trên cây suy biến -> RecursionError:", str(e)[:50])
it = BSTIteratorArrayStack(left_chain(10**5))
print("Cách 3 (stack) chạy tốt, next() =", it.next())

Cách 2 (đệ quy) trên cây suy biến -> RecursionError: maximum recursion depth exceeded
Cách 3 (stack) chạy tốt, next() = 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Iterator trên BST** ⇒ inorder. Muốn **lười** ⇒ stack "đẩy hết nhánh trái" (khuôn LC 173).
- **Hai chiều (`prev`)** ⇒ thêm **bộ đệm** các phần tử đã sinh + chỉ số; `prev` không bao giờ cần sinh mới.
- Mô hình con trỏ "giữa các phần tử": khởi đầu `i = -1`; `hasNext ⇔ i < len-1 (hoặc còn nguồn)`; `hasPrev ⇔ i > 0`.
- Cây có thể sâu `10⁵` ⇒ tránh đệ quy trong Python.
- Kiểm tra nhanh: cây 1 nút ⇒ `hasPrev` luôn `False`; sau `next` đầu tiên `hasPrev` vẫn `False`.

## 15. Biến thể của bài

1. **LC 173 — Binary Search Tree Iterator:** chỉ `next`/`hasNext`, yêu cầu bộ nhớ `O(h)` ⇒ chính là phần "nguồn sinh" của cách 4.
2. **Iterator giảm dần:** đẩy hết nhánh **phải**, pop rồi chuyển sang cây con **trái** (bài tập 4).
3. **LC 653 — Two Sum IV (BST):** dùng **hai** iterator (tăng và giảm) như hai con trỏ trên mảng đã sắp, bộ nhớ `O(h)` (bài tập 5).
4. **LC 285/510 — Inorder Successor:** tìm phần tử kế tiếp của một nút (có/không có con trỏ cha).
5. **Iterator hai chiều bộ nhớ `O(h)`:** không lưu `arr`, mà tìm predecessor bằng một stack thứ hai hoặc con trỏ cha — phức tạp hơn, đổi bộ nhớ lấy thời gian.
6. **LC 284 — Peeking Iterator:** thêm `peek()` không di chuyển con trỏ (bài tập 3).

## 16. So sánh các cách

| Cách | Ý tưởng | Time mỗi thao tác | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | duyệt lại cả cây mỗi lần | `O(n)` | `O(n)` tạm | oracle kiểm thử |
| 2. Mảng (đệ quy) | inorder đệ quy một lần + chỉ số | `O(1)` (khởi tạo `O(n)`) | `O(n)` | cây nông, code ngắn nhất |
| 3. Mảng (stack) | inorder bằng stack trạng thái + chỉ số | `O(1)` (khởi tạo `O(n)`) | `O(n)` | an toàn với cây sâu |
| 4. Lười | stack sinh dần + `arr` đã sinh + chỉ số | khấu hao `O(1)` (khởi tạo `O(h)`) | `O(h + k)` | follow-up; chỉ duyệt một phần cây |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** Cây `[4, 2, 6, 1, 3, 5, 7]`, chuỗi lệnh `next, next, next, prev, prev, hasPrev, next, hasNext`. Viết mảng kết quả (bỏ `null` của khởi tạo).

**Bài 2 (tính tay):** Với cây `[4, 2, 6, 1, 3, 5, 7]` và cách 4, viết nội dung `stack` (giá trị, đáy → đỉnh) **sau khởi tạo** và **sau mỗi** trong 4 lần `next()` đầu tiên.

**Bài 3 (điền TODO):** Thêm phương thức `peekNext()` và `peekPrev()` vào bản lười: trả về giá trị tiếp theo / trước đó **mà không di chuyển** con trỏ (giả sử luôn hợp lệ).

**Bài 4 (tự viết lại):** Viết `ReverseBSTIterator` duyệt **giảm dần** (lười), với `hasNext/next` (không cần `prev`).

**Bài 5 (mini challenge — LC 653):** Viết `ex5_find_target(root, k)`: có tồn tại hai nút **khác nhau** có tổng bằng `k` không? Dùng hai iterator (tăng và giảm), bộ nhớ `O(h)`.

In [22]:
# Bài 3 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
class Ex3PeekIterator(BSTIteratorLazy):
    def peekNext(self) -> int:
        # TODO: nếu i + 1 < len(arr) thì trả arr[i + 1]
        # TODO: ngược lại, phần tử kế tiếp là đỉnh stack (chưa pop)
        pass

    def peekPrev(self) -> int:
        # TODO: trả arr[i - 1]
        pass

In [23]:
# Bài 4 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
class Ex4ReverseBSTIterator:
    def __init__(self, root):
        self.stack = []
        # TODO: đẩy hết nhánh PHẢI từ root

    def hasNext(self):
        # TODO
        pass

    def next(self):
        # TODO: pop, rồi đẩy hết nhánh phải của cây con TRÁI
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Inorder: `[1..7]`. (2) Theo dõi `i` sau mỗi lệnh. (3) Sau `next×3` thì `i = 2`; `prev×2` thì `i = 0`.

**Bài 2.** (1) Khởi tạo đẩy `4, 2, 1`. (2) `next` pop 1 (không con phải). (3) Pop 2 rồi đẩy nhánh trái của 3 (chỉ 3); pop 3; pop 4 rồi đẩy `6, 5`.

**Bài 3.** (1) Phần tử kế tiếp có thể đã nằm trong `arr` (sau `prev`). (2) Nếu chưa, đó là `stack[-1].val` — không cần pop. (3) `peekPrev` luôn là `arr[i - 1]`.

**Bài 4.** (1) Đối xứng LC 173: thay `left` bằng `right` và ngược lại. (2) `hasNext` ⇔ stack khác rỗng. (3) Kiểm tra bằng `sorted(vals, reverse=True)`.

**Bài 5.** (1) Iterator tăng cho con trỏ trái, giảm cho con trỏ phải. (2) Lặp khi `a < b`: tổng nhỏ ⇒ tiến trái, lớn ⇒ tiến phải. (3) Giá trị trong BST phân biệt nên `a < b` đảm bảo hai nút khác nhau.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1
cmds1 = ["BSTIterator", "next", "next", "next", "prev", "prev", "hasPrev", "next", "hasNext"]
args1 = [[[4, 2, 6, 1, 3, 5, 7]]] + [[] for _ in range(8)]
ans1 = run_commands(BSTIterator, cmds1, args1)[1:]
print("Bài 1:", ans1)
assert ans1 == [1, 2, 3, 2, 1, False, 2, True]

# Bài 2
it2 = BSTIteratorLazy(build_tree([4, 2, 6, 1, 3, 5, 7]))
states = [[n.val for n in it2.stack]]
for _ in range(4):
    it2.next()
    states.append([n.val for n in it2.stack])
print("Bài 2:", states)
assert states == [[4, 2, 1], [4, 2], [4, 3], [4], [6, 5]]
print("Bài 1, 2: OK")

Bài 1: [1, 2, 3, 2, 1, False, 2, True]
Bài 2: [[4, 2, 1], [4, 2], [4, 3], [4], [6, 5]]
Bài 1, 2: OK


In [25]:
# Bài 3
class Ex3PeekIteratorSol(BSTIteratorLazy):
    def peekNext(self) -> int:
        if self.i + 1 < len(self.arr):
            return self.arr[self.i + 1]
        return self.stack[-1].val

    def peekPrev(self) -> int:
        return self.arr[self.i - 1]

rng = random.Random(3)
for _ in range(400):
    vals = rng.sample(range(100), rng.randint(1, 20))
    sv3 = sorted(vals)
    it = Ex3PeekIteratorSol(random_bst(vals))
    i = -1
    for _ in range(40):
        if it.hasNext():
            assert it.peekNext() == sv3[i + 1]
        if it.hasPrev():
            assert it.peekPrev() == sv3[i - 1]
        if rng.random() < 0.6 and it.hasNext():
            i += 1
            assert it.next() == sv3[i]
        elif it.hasPrev():
            i -= 1
            assert it.prev() == sv3[i]
print("Bài 3: OK")

Bài 3: OK


In [26]:
# Bài 4
class Ex4ReverseBSTIteratorSol:
    def __init__(self, root):
        self.stack = []
        self._push_right(root)

    def _push_right(self, node):
        while node:
            self.stack.append(node)
            node = node.right

    def hasNext(self):
        return bool(self.stack)

    def next(self):
        node = self.stack.pop()
        self._push_right(node.left)
        return node.val

rng = random.Random(4)
for _ in range(300):
    vals = rng.sample(range(1000), rng.randint(1, 30))
    it = Ex4ReverseBSTIteratorSol(random_bst(vals))
    got = []
    while it.hasNext():
        got.append(it.next())
    assert got == sorted(vals, reverse=True)
print("Bài 4: OK, phần tử đầu tiên (lớn nhất) của cây ví dụ:", Ex4ReverseBSTIteratorSol(build_tree([7, 3, 15, None, None, 9, 20])).next())

Bài 4: OK, phần tử đầu tiên (lớn nhất) của cây ví dụ: 20


In [27]:
# Bài 5 — LC 653 với hai iterator
class _Forward:
    def __init__(self, root):
        self.stack = []
        self._push(root)

    def _push(self, node):
        while node:
            self.stack.append(node)
            node = node.left

    def next(self):
        node = self.stack.pop()
        self._push(node.right)
        return node.val

def ex5_find_target(root, k):
    lo, hi = _Forward(root), Ex4ReverseBSTIteratorSol(root)
    a, b = lo.next(), hi.next()
    while a < b:
        s = a + b
        if s == k:
            return True
        if s < k:
            a = lo.next()
        else:
            b = hi.next()
    return False

def ex5_brute(vals, k):
    return any(vals[i] + vals[j] == k for i in range(len(vals)) for j in range(i + 1, len(vals)))

t5 = build_tree([5, 3, 6, 2, 4, None, 7])
print("Bài 5:", ex5_find_target(t5, 9), ex5_find_target(t5, 28))
assert ex5_find_target(t5, 9) is True and ex5_find_target(t5, 28) is False
rng = random.Random(5)
for _ in range(500):
    vals = rng.sample(range(50), rng.randint(1, 15))
    k = rng.randint(0, 100)
    assert ex5_find_target(random_bst(vals), k) == ex5_brute(vals, k)
print("Bài 5: OK")

Bài 5: True False
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
class BSTIterator:
    def __init__(self, root):
        self.stack, self.arr, self.i = [], [], -1
        self._push_left(root)

    def _push_left(self, node):
        while node:
            self.stack.append(node)
            node = node.left

    def hasNext(self):
        return self.i < len(self.arr) - 1 or bool(self.stack)

    def next(self):
        self.i += 1
        if self.i == len(self.arr):              # sinh phần tử mới
            node = self.stack.pop()
            self._push_left(node.right)
            self.arr.append(node.val)
        return self.arr[self.i]

    def hasPrev(self):
        return self.i > 0

    def prev(self):
        self.i -= 1
        return self.arr[self.i]
```

**Checklist trước khi submit:**
- [ ] Tên lớp/phương thức: `BSTIterator`, `hasNext`, `next`, `hasPrev`, `prev`.
- [ ] Con trỏ khởi đầu `i = -1`.
- [ ] `hasPrev` là `i > 0` (không phải `>= 0`).
- [ ] Bản lười: `hasNext` xét **cả** `arr` phía sau con trỏ **và** stack; `next` dùng lại `arr` khi `i < len(arr) - 1`.
- [ ] Không dùng đệ quy sâu (cây có thể suy biến `10⁵` tầng).
- [ ] Thử nhanh: ví dụ đề; cây 1 nút; `next, next, prev, next` trên cây ví dụ → `3, 7, 3, 7`.